In [53]:
!pip install scikit-learn==1.6.1 joblib pandas fastapi uvicorn pydantic -q

# Safety net: if scikit-learn was already imported with a WRONG version
# earlier in this session, force-restart so the install above actually takes effect.
import sys
if "sklearn" in sys.modules:
    import sklearn
    if sklearn.__version__ != "1.6.1":
        import os
        print("Wrong sklearn version was already loaded — restarting runtime automatically.")
        os.kill(os.getpid(), 9)

In [54]:
from google.colab import files
uploaded = files.upload()   # click "Choose Files", select heart_disease_final_model_v1.pkl, wait for 100%

import os
if os.path.exists("heart_disease_final_model_v1.pkl") and not os.path.exists("model.pkl"):
    os.rename("heart_disease_final_model_v1.pkl", "model.pkl")

print(os.listdir())
assert os.path.exists("model.pkl"), "model.pkl still missing — check the filename you uploaded matches exactly"

import sklearn
print("scikit-learn version:", sklearn.__version__)
assert sklearn.__version__ == "1.6.1", "Wrong version — restart runtime manually and re-run Cell 1."

import joblib
model = joblib.load("model.pkl")
print(model)  # should print the full Pipeline with no error

Saving .json to .json (3)
Saving model.pkl to model (3).pkl
['model (1).pkl', '.json', 'predict.py', 'model.pkl', '.gitignore', '.json (1)', 'tests', 'docs', '__pycache__', 'model (2).pkl', '.json (3)', '.git', '.json (2)', 'requirements.txt', 'pyproject.toml', 'uvicorn.log', 'models', 'README.md', 'research', 'model (3).pkl', 'app', 'test_pipeline.py', '.github', 'src']
scikit-learn version: 1.6.1
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
      

In [55]:
%%writefile main.py
import os
import logging
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, Field
from predict import predict_heart_disease, load_model

logger = logging.getLogger("heart_disease_api")
app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

api_key_header = APIKeyHeader(name="X-API-Key")
SECRET_KEY = os.environ.get("HEART_API_KEY", "dev-only-change-me")

def verify_api_key(api_key: str = Depends(api_key_header)) -> str:
    if api_key != SECRET_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return api_key

class HeartData(BaseModel):
    age: float
    sex: int = Field(..., ge=0, le=1)
    cp: int = Field(..., ge=0, le=3)
    trestbps: float
    chol: float
    fbs: int = Field(..., ge=0, le=1)
    restecg: int = Field(..., ge=0, le=2)
    thalach: float
    exang: int = Field(..., ge=0, le=1)
    oldpeak: float
    slope: int = Field(..., ge=0, le=2)
    ca: int = Field(..., ge=0, le=4)
    thal: int

@app.on_event("startup")
def _warm_up_model():
    load_model()
    logger.info("Startup complete: model pre-loaded, API ready.")

@app.get("/health")
def health_check():
    return {"status": "ok", "model": "SVM (UCI Heart Disease)"}

@app.post("/predict")
def get_prediction(data: HeartData, key: str = Depends(verify_api_key)):
    try:
        return predict_heart_disease(data.dict())
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except Exception:
        logger.exception("Unhandled error during prediction")
        raise HTTPException(status_code=500, detail="Internal prediction error")

Writing main.py


In [56]:
import os, sys

problems = []

# 1. Model file present?
if not os.path.exists("model.pkl"):
    problems.append("model.pkl is missing. Re-upload it and rename it to model.pkl (Cell 2).")

# 2. Correct sklearn version?
import sklearn
if sklearn.__version__ != "1.6.1":
    problems.append(f"scikit-learn is {sklearn.__version__}, need 1.6.1. Re-run Cell 1 and restart the runtime.")

# 3. Code files present?
for f in ["predict.py", "main.py"]:
    if not os.path.exists(f):
        problems.append(f"{f} is missing. Re-run its %%writefile cell.")

if problems:
    print("Fix these before launching the server:")
    for p in problems:
        print(" -", p)
else:
    print("Preflight passed: safe to launch uvicorn.")

Preflight passed: safe to launch uvicorn.


In [57]:
import os
print("Current folder:", os.getcwd())
print(os.listdir("."))

Current folder: /content/GROUP6_REPOSITORY-
['model (1).pkl', '.json', 'predict.py', 'model.pkl', '.gitignore', '.json (1)', 'tests', 'docs', '__pycache__', 'model (2).pkl', 'main.py', '.json (3)', '.git', '.json (2)', 'requirements.txt', 'pyproject.toml', 'uvicorn.log', 'models', 'README.md', 'research', 'model (3).pkl', 'app', 'test_pipeline.py', '.github', 'src']


In [58]:
from main import app

In [59]:
%%writefile predict.py
import logging
import time
import joblib
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("heart_disease_inference")

NUMERIC_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
REQUIRED_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

_model_cache = None

def load_model(model_path: str = "model.pkl"):
    global _model_cache
    if _model_cache is None:
        logger.info(f"Loading model from '{model_path}'...")
        _model_cache = joblib.load(model_path)
        logger.info("Model loaded and cached successfully.")
    return _model_cache

def validate_input(input_data: dict) -> None:
    missing = [f for f in REQUIRED_FEATURES if f not in input_data]
    if missing:
        raise ValueError(f"Missing required feature(s): {missing}")
    unexpected = [f for f in input_data if f not in REQUIRED_FEATURES]
    if unexpected:
        logger.warning(f"Ignoring unexpected field(s) not used by the model: {unexpected}")

def predict_heart_disease(input_data: dict) -> dict:
    start_time = time.time()
    validate_input(input_data)
    model = load_model()
    df = pd.DataFrame([{k: input_data[k] for k in REQUIRED_FEATURES}])
    try:
        prediction = int(model.predict(df)[0])
        proba = model.predict_proba(df)[0]
        confidence = float(proba[prediction])
    except Exception:
        logger.exception("Inference failed on input payload.")
        raise
    latency_ms = (time.time() - start_time) * 1000
    result = {
        "prediction": prediction,
        "label": "Disease" if prediction == 1 else "No Disease",
        "probability": round(confidence, 4),
        "latency_ms": round(latency_ms, 2),
    }
    logger.info(f"Prediction={result['prediction']} ({result['label']}) | confidence={result['probability']} | latency={result['latency_ms']}ms")
    return result

Overwriting predict.py


In [60]:
%%writefile test_pipeline.py
from predict import predict_heart_disease, REQUIRED_FEATURES

dummy_payload = {
    "age": 55, "sex": 1, "cp": 3, "trestbps": 140, "chol": 250,
    "fbs": 0, "restecg": 1, "thalach": 150, "exang": 0, "oldpeak": 2.0,
    "slope": 1, "ca": 0, "thal": 2,
}

print("Test 1: normal payload")
result = predict_heart_disease(dummy_payload)
assert result["prediction"] in [0, 1]
assert "latency_ms" in result
print("  passed:", result)

print("\nTest 2: missing feature should raise ValueError")
bad_payload = dummy_payload.copy()
del bad_payload["thal"]
try:
    predict_heart_disease(bad_payload)
    raise AssertionError("Expected ValueError, got none")
except ValueError as e:
    print("  passed — correctly rejected:", e)

print(f"\nAll checks passed. Model expects {len(REQUIRED_FEATURES)} features.")

Overwriting test_pipeline.py


In [62]:
import os
os.environ["HEART_API_KEY"] = "choose-a-real-secret-here"  # <-- change this to your own secret

import subprocess, time

log_file = open("uvicorn.log", "w")
proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(8)
log_file.flush()
!cat uvicorn.log
print("Process still running:", proc.poll() is None)

INFO:     Started server process [8056]
INFO:     Waiting for application startup.
2026-09-28 04:00:53,285 - INFO - Loading model from 'model.pkl'...
2026-09-28 04:00:54,447 - INFO - Model loaded and cached successfully.
2026-09-28 04:00:54,447 - INFO - Startup complete: model pre-loaded, API ready.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Process still running: True


In [63]:
!python test_pipeline.py

Test 1: normal payload
2026-09-28 04:01:10,400 - INFO - Loading model from 'model.pkl'...
2026-09-28 04:01:11,145 - INFO - Model loaded and cached successfully.
2026-09-28 04:01:11,165 - INFO - Prediction=0 (No Disease) | confidence=0.7593 | latency=764.63ms
  passed: {'prediction': 0, 'label': 'No Disease', 'probability': 0.7593, 'latency_ms': 764.63}

Test 2: missing feature should raise ValueError
  passed — correctly rejected: Missing required feature(s): ['thal']

All checks passed. Model expects 13 features.


In [64]:
!curl http://localhost:8000/health

{"status":"ok","model":"SVM (UCI Heart Disease)"}

In [65]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [66]:
import subprocess, time, re

cf_log = open("cloudflared.log", "w")
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=cf_log, stderr=subprocess.STDOUT,
)
time.sleep(8)
cf_log.flush()
log_content = open("cloudflared.log").read()
match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_content)

if match:
    public_url = match.group(0)
    print("Give your teammate this base URL for the predict endpoint:")
    print(f"{public_url}/predict")
else:
    print("URL not found yet — run this cell again, or check cloudflared.log below:")
    print(log_content)

Give your teammate this base URL for the predict endpoint:
https://rendered-units-land-bloomberg.trycloudflare.com/predict


In [67]:
# 1. Install pyngrok and dependencies
!pip install pyngrok uvicorn fastapi nest_asyncio

from pyngrok import ngrok
import uvicorn
import os
import sys
import nest_asyncio
import threading

# Patch asyncio to support running uvicorn within Colab
nest_asyncio.apply()

# 1. Force kill existing processes
!pkill -9 ngrok
!pkill -9 uvicorn

# 2. Clone the repo if it's not present, or pull latest changes
REPO_DIR = "GROUP6_REPOSITORY-"
REPO_URL = "https://github.com/stacey017/GROUP6_REPOSITORY-.git"

if not os.path.exists(REPO_DIR):
    !git clone "{REPO_URL}"
else:
    %cd {REPO_DIR}
    !git pull origin main
    %cd ..

# 3. Enter the repo directory
os.chdir(f"/content/{REPO_DIR}")
sys.path.insert(0, os.getcwd())

print("--------------------------------------------------")
print("✅ Current Directory:", os.getcwd())
print("📁 Files Found:", os.listdir("."))
print("--------------------------------------------------")

# 4. Setup ngrok
ngrok.set_auth_token("3JuTKsdhfofO19Lt2f0Ysqe9xPl_3jbx5cUet9HRCH6jsyFXN")

# Disconnect old tunnels
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# 5. Connect public tunnel
public_url = ngrok.connect(8000)

print("\n--------------------------------------------------")
print("🚀 LIVE PUBLIC ENDPOINT:", public_url.public_url)
print("📖 INTERACTIVE SWAGGER UI:", f"{public_url.public_url}/docs")
print("--------------------------------------------------\n")

# 6. Import FastAPI app and run in a background thread to prevent blocking the event loop
from main import app  # <-- change to "from main import app" if main.py is at repo root, not inside api/

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Run the server in a separate thread
thread = threading.Thread(target=server.run)
thread.start()

Cloning into 'GROUP6_REPOSITORY-'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 360 (delta 68), reused 116 (delta 21), pack-reused 135 (from 1)
Receiving objects: 100% (360/360), 3.88 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (100/100), done.
--------------------------------------------------
✅ Current Directory: /content/GROUP6_REPOSITORY-
📁 Files Found: ['model (1).pkl', '.json', 'predict.py', 'model.pkl', '.gitignore', '.json (1)', 'tests', 'docs', '__pycache__', 'model (2).pkl', 'main.py', '.json (3)', '.git', '.json (2)', 'requirements.txt', 'pyproject.toml', 'uvicorn.log', 'models', 'cloudflared', 'README.md', 'research', 'GROUP6_REPOSITORY-', 'model (3).pkl', 'app', 'test_pipeline.py', 'cloudflared.log', '.github', 'src']
--------------------------------------------------

--------------------------------------------------
🚀 LIVE PUBLIC ENDPOINT: https://lyricist-s

INFO:     Started server process [3145]
INFO:     Waiting for application startup.


In [68]:
import os
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if not d.startswith(".git")]
    for f in files:
        if f == "main.py":
            print(os.path.join(root, f))

./main.py


In [69]:
print("Thread alive:", thread.is_alive())

Thread alive: True


In [70]:
print(ngrok.get_tunnels())

[<NgrokTunnel: "https://lyricist-seizing-uproot.ngrok-free.dev" -> "http://localhost:8000">]


In [71]:
!curl http://localhost:8000/health

INFO:     127.0.0.1:38610 - "GET /health HTTP/1.1" 200 OK
{"status":"ok","model":"SVM (UCI Heart Disease)"}

In [72]:
for route in app.routes:
    print(route.path)

/openapi.json
/docs
/docs/oauth2-redirect
/redoc
/health
/predict


In [73]:
!curl http://localhost:8000/<actual-path-from-step-1>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `curl http://localhost:8000/<actual-path-from-step-1>'
